# ROOT tree generation using PYTHIA8

At first, we import the necessary libraries.

In [25]:
import ROOT
import pythia8

from array import array

Then, we configure PYTHIA8 for $\sqrt{s_\mathrm{NN}} = 510 \: \mathrm{GeV}$ pp collisions and use the Detroit tune for RHIC.

In [26]:
pythia = pythia8.Pythia()

pythia.readString("Beams:idA = 2212") # proton
pythia.readString("Beams:idB = 2212") # proton
pythia.readString("Beams:eCM = 510.") # collision energy in GeV

pythia.readString("hardQCD:all = on") # enable hard QCD processes
pythia.readString("PhaseSpace:pTHatMin = 20.") # minimum pT of scattered partons in GeV

pythia.readString("PartonLevel:MPI = on") # enable multi-parton interactions
pythia.readString("PartonLevel:ISR = on") # enable initial state radiation
pythia.readString("PartonLevel:FSR = on") # enable final state radiation
pythia.readString("HadronLevel:Decay = on") # enable decay of hadronization products

pythia.readString("Tune:pp = 33") # the Detroit tune for RHIC

True


 *------------------------------------------------------------------------------------* 
 |                                                                                    | 
 |  *------------------------------------------------------------------------------*  | 
 |  |                                                                              |  | 
 |  |                                                                              |  | 
 |  |   PPP   Y   Y  TTTTT  H   H  III    A      Welcome to the Lund Monte Carlo!  |  | 
 |  |   P  P   Y Y     T    H   H   I    A A     This is PYTHIA version 8.312      |  | 
 |  |   PPP     Y      T    HHHHH   I   AAAAA    Last date of change: 23 May 2024  |  | 
 |  |   P       Y      T    H   H   I   A   A                                      |  | 
 |  |   P       Y      T    H   H  III  A   A    Now is 13 Mar 2026 at 23:21:51    |  | 
 |  |                                                                              |  | 
 |  |   Program docu

Now, we prepare the ROOT tree with branches which we need to fill later with event data.

In [27]:
file = ROOT.TFile("events.root", "RECREATE") # create a .root file
tree = ROOT.TTree("events", "A tree with pp events from PYTHIA8") # create the tree

# allocate memory for observables
Nch = array('i', [0]) # 1D array for multiplicity (single precision)
pT_vector = ROOT.std.vector('double')() # vector for pT (double precision)

# create branches
tree.Branch("multiplicity", Nch, "multiplicity/I")
tree.Branch("pT", pT_vector)

Event generation:

In [28]:
pythia.readString("Random:setSeed = on") # enable random seed setting
pythia.readString("Random:seed = 67") # set random seed to 0 (random every time) or to a specific value for reproducibility

pythia.init() # initialize PYTHIA8

nEvents = 1000 # set the number of events
for iEvent in range(nEvents): # event loop
    
    Nch[0] = 0
    pT_vector.clear() # clear the pT vector for each event

    if not pythia.next(): # generate an event
        continue

    nCharged = 0 # define the multiplicity variable
    for i in range(pythia.event.size()): # particle loop
        particle = pythia.event[i]
        if particle.isFinal() and particle.isCharged() and abs(particle.eta())<1:
            nCharged += 1
            pT_vector.push_back(particle.pT())
    Nch[0] = nCharged

    tree.Fill()


 *-------  PYTHIA Process Initialization  --------------------------*
 |                                                                  |
 | We collide p+ with p+ at a CM energy of 5.100e+02 GeV            |
 |                                                                  |
 |------------------------------------------------------------------|
 |                                                    |             |
 | Subprocess                                    Code |   Estimated |
 |                                                    |    max (mb) |
 |                                                    |             |
 |------------------------------------------------------------------|
 |                                                    |             |
 | g g -> g g                                     111 |   3.113e-03 |
 | g g -> q qbar (uds)                            112 |   7.506e-05 |
 | q g -> q g                                     113 |   6.614e-03 |
 | q q(bar)' -> q q

Finally, we save everything into a .root file.

In [29]:
tree.Write()
file.Close()